# BICAM Congressional Hearings — Data Exploration

Key questions from the setup guide:
- What does the transcript text field look like?
- Are speaker names embedded in the transcript?
- How does the format compare to the original paper's source files?
- Any data quality issues?

In [1]:
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="python-dotenv")

# Navigate to project root so 'src' is importable
project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if not os.path.isdir(os.path.join(project_root, "src")):
    # Fallback: assume cwd is project root or notebooks/
    if os.path.isdir("src"):
        project_root = os.getcwd()
    else:
        project_root = os.path.dirname(os.getcwd())
os.chdir(project_root)
sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

import pandas as pd

from src.data import TEXT_COLUMN, filter_hearings_by_congress, load_hearings, load_hearings_texts_chunked
from src.preprocess import create_sentence_records, download_nltk_deps, segment_speakers

download_nltk_deps()
print("Ready.")

Project root: /Users/maximgerman/Private/Uni/Deep Learning/congressional-empirical-discourse


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 13
python-dotenv could not parse statement starting at line 14
python-dotenv could not parse statement starting at line 15
python-dotenv could not parse statement starting at line 16
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 18
python-dotenv could not parse statement starting

Ready.


## 1. Hearings Metadata

In [2]:
hearings = load_hearings()
print(f"Total hearings: {len(hearings)}")
print(f"Columns: {list(hearings.columns)}")
print(f"Congress range: {hearings['congress'].min()} - {hearings['congress'].max()}")
print("\nHearings per congress:")
hearings["congress"].value_counts().sort_index()

Total hearings: 45432
Columns: ['hearing_id', 'hearing_jacketnumber', 'loc_id', 'title', 'congress', 'session', 'chamber', 'is_appropriation', 'hearing_number', 'part_number', 'citation', 'pages', 'updated_at']
Congress range: 79 - 118

Hearings per congress:


congress
79       33
82        3
84       24
85        8
86        1
87      623
88      552
89      677
90      693
91      820
92      980
93     1115
94     1291
95     1397
96     1377
97      131
99        2
100       1
102       5
103     188
104     395
105     864
106    1859
107    2335
108    2489
109    2977
110    3752
111    3445
112    3281
113    2909
114    2824
115    2571
116    2182
117    2132
118    1496
Name: count, dtype: int64

In [3]:
# Filter to our target: 115th+ Congress, House only
new_era = filter_hearings_by_congress(hearings, min_congress=115, chamber="house")
print(f"115th+ House hearings: {len(new_era)}")
print("\nBreakdown by congress:")
new_era["congress"].value_counts().sort_index()

115th+ House hearings: 5070

Breakdown by congress:


congress
115    1468
116    1429
117    1116
118    1057
Name: count, dtype: int64

## 2. Transcript Format Inspection

Load a few transcripts and inspect the raw text format.

In [4]:
target_ids = new_era[new_era["congress"] == 115]["hearing_id"].head(5).tolist()
texts = load_hearings_texts_chunked(target_ids)
print(f"Loaded {len(texts)} transcripts")
print(f"Columns: {list(texts.columns)}")
texts.head()

Loaded 5 transcripts
Columns: ['hearing_id', 'raw_text', 'pdf', 'formatted_text']


,hearing_id,raw_text,pdf,formatted_text
0,hhrg23827-115,"\n[House Hearing, 115 Congress]\n[From the U.S...",https://congress.gov/115/chrg/CHRG-115hhrg2382...,https://congress.gov/115/chrg/CHRG-115hhrg2382...
1,hhrg23844-115,"\n[House Hearing, 115 Congress]\n[From the U.S...",https://congress.gov/115/chrg/CHRG-115hhrg2384...,https://congress.gov/115/chrg/CHRG-115hhrg2384...
2,hhrg29473-115,"\n[House Hearing, 115 Congress]\n[From the U.S...",https://congress.gov/115/chrg/CHRG-115hhrg2947...,https://congress.gov/115/chrg/CHRG-115hhrg2947...
3,hhrg31509-115,"\n[House Hearing, 115 Congress]\n[From the U.S...",https://congress.gov/115/chrg/CHRG-115hhrg3150...,https://congress.gov/115/chrg/CHRG-115hhrg3150...
4,hhrg33477-115,"\n[House Hearing, 115 Congress]\n[From the U.S...",https://congress.gov/115/chrg/CHRG-115hhrg3347...,https://congress.gov/115/chrg/CHRG-115hhrg3347...


In [5]:
# Show first 2000 chars of a sample transcript
row = texts[texts[TEXT_COLUMN].str.len() > 100].iloc[0]
text = row[TEXT_COLUMN]
hearing_id = row["hearing_id"]

print(f"Hearing: {hearing_id}")
print(f"Text length: {len(text):,} chars")
print("=" * 60)
print(text[:2000])

Hearing: hhrg23827-115
Text length: 150,005 chars

[House Hearing, 115 Congress]
[From the U.S. Government Publishing Office]





      HELPING STUDENTS SUCCEED THROUGH THE POWER OF SCHOOL CHOICE


                                HEARING

                               before the

                    SUBCOMMITTEE ON EARLY CHILDHOOD,
                  ELEMENTARY, AND SECONDARY EDUCATION

                         COMMITTEE ON EDUCATION
                           AND THE WORKFORCE

                     U.S. House of Representatives

                     ONE HUNDRED FIFTEENTH CONGRESS

                             FIRST SESSION

                               __________

            HEARING HELD IN WASHINGTON, DC, FEBRUARY 2, 2017

                               __________

                            Serial No. 115-2

                               __________

  Printed for the use of the Committee on Education and the Workforce



[GRAPHIC(S) NOT AVAILABLE IN TIFF FORMAT]






        

## 3. Speaker Segmentation

Test splitting the transcript into per-speaker chunks.

In [6]:
chunks = segment_speakers(text)
print(f"Speaker chunks found: {len(chunks)}")
print(f"Unique speakers: {len(set(c['speaker'] for c in chunks))}")
print()
for i, c in enumerate(chunks[:8]):
    print(f"[{i}] {c['speaker']}: {c['text'][:100]}...")

Speaker chunks found: 223
Unique speakers: 20

[0] Chairman Rokita: Good morning. A quorum being present the subcommittee on Early Childhood, Elementary, and Secondary ...
[1] Mr. Polis: Thank you....
[2] Chairman Rokita: In recent years, this subcommittee has helped advance positive legislative solutions for America's s...
[3] Mr. Polis: I want to thank Chairman Rokita. And in particular, I wanted to thank him for delaying the opening o...
[4] Chairman Rokita: I thank the gentleman. Pursuant to committee Rule 7(c), all members will be permitted to submit writ...
[5] Chairman Rokita: Please let the record reflect that all witnesses answered in the affirmative. Before I recognize you...
[6] Mr. Williams: Good morning, Chairman Rokita, Ranking Member Polis, and I see Chairwoman Foxx and members of the co...
[7] Chairman Rokita: Thank you, Mr. Williams. Ms. Carter, you are recognized for 5 minutes. TESTIMONY OF ALMO J. CARTER, ...


## 4. Sentence Splitting with Context Windows

Split speaker chunks into individual sentences with before/after context.

In [7]:
records = create_sentence_records(chunks[:3], hearing_id)
print(f"Sentences from first 3 speaker chunks: {len(records)}")
print()
for r in records[:5]:
    print(f"Speaker:  {r['speaker']} (last_name: {r['speaker_last_name']})")
    print(f"Before:   {r['context_before'][:80]}")
    print(f"Target:   {r['target_sentence'][:80]}")
    print(f"After:    {r['context_after'][:80]}")
    print()

Sentences from first 3 speaker chunks: 97

Speaker:  Chairman Rokita (last_name: ROKITA)
Before:   
Target:   Good morning.
After:    A quorum being present the subcommittee on Early Childhood, Elementary, and Seco

Speaker:  Chairman Rokita (last_name: ROKITA)
Before:   Good morning.
Target:   A quorum being present the subcommittee on Early Childhood, Elementary, and Seco
After:    Welcome to the first hearing of the subcommittee for the 115th Congress.

Speaker:  Chairman Rokita (last_name: ROKITA)
Before:   A quorum being present the subcommittee on Early Childhood, Elementary, and Seco
Target:   Welcome to the first hearing of the subcommittee for the 115th Congress.
After:    I thank everyone for their cooperation.

Speaker:  Chairman Rokita (last_name: ROKITA)
Before:   Welcome to the first hearing of the subcommittee for the 115th Congress.
Target:   I thank everyone for their cooperation.
After:    We are starting a little bit late, only to accommodate the prayer breakfast hel

## 5. Pipeline Output

Inspect the pre-built pipeline outputs (if available).

In [8]:
sample_path = os.path.join(os.getcwd(), "data", "sample_for_labeling.csv")
if os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    print(f"Silver labeling sample: {len(sample):,} sentences")
    print("\nCongress distribution:")
    print(sample["congress"].value_counts().sort_index())
    print("\nParty distribution:")
    print(sample["party"].value_counts())
    print("\nMinority status (1=minority, 0=majority):")
    print(sample["minority"].value_counts())
    print("\nSample rows:")
    display(
        sample[["hearing_id", "speaker", "target_sentence", "congress", "party", "minority", "committee_name"]].head(10)
    )
else:
    print("No pipeline output found. Run: .venv/bin/python -m src.pipeline")

Silver labeling sample: 10,000 sentences

Congress distribution:
congress
115    2500
116    2500
117    2500
118    2500
Name: count, dtype: int64

Party distribution:
party
Republican     5137
Democratic     4850
Independent      12
Libertarian       1
Name: count, dtype: int64

Minority status (1=minority, 0=majority):
minority
0.0    6469
1.0    3531
Name: count, dtype: int64

Sample rows:


,hearing_id,speaker,target_sentence,congress,party,minority,committee_name
0,hhrg31342-115,Mr. Bost,"Thank you, Mr. Chairman.",115,Republican,0.0,Committee on Veterans' Affairs
1,hhrg24919-115,Mr. King,"And the answer was, well, her answer was that ...",115,Republican,0.0,Committee on Agriculture
2,hhrg24674-115,Chairman BLUM,"We maintain open spaces, healthy rangelands, p...",115,Republican,0.0,Committee on Small Business
3,hhrg27069-115,Mr. LAWSON,We must do better.,115,Democratic,1.0,Committee on Small Business
4,hhrg32389-115,Ms. Castor,"Under the Affordable Care Act, under your priv...",115,Democratic,1.0,Committee on Energy and Commerce
5,hhrg25147-115,Mr. Rogers,How many of these recommendations were impleme...,115,Republican,0.0,Committee on Foreign Affairs
6,hhrg31368-115,Mr. Gianforte,And there are just some silly impediments to g...,115,Republican,0.0,Committee on Oversight and Government Reform
7,hhrg29688-115,Mr. Bergman,"Mr. Chairman, this concludes my testimony.",115,Republican,0.0,Committee on Veterans' Affairs
8,hhrg24442-115,Ms. Jayapal,Dr. Book.,115,Democratic,1.0,Committee on the Budget
9,hhrg27511-115,Mrs. Wagner,"Mr. Mastic, you wrote that advancing women's e...",115,Republican,0.0,NaN
